# Canada Coverage Analysis

End-to-end demonstration of **missiontools** coverage and plotting features.

**Scenario**
- Sun-synchronous orbit, 550 km, 10:30 LTAN (ascending)
- Nadir-pointed pushbroom sensor, 20° half-angle FOV
- Min ground elevation: 20°
- Illumination constraint: solar zenith angle < 70° (daytime)
- Area of interest: Canada (20 000 km² point density)
- Simulation window: 14 days

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs

from missiontools import Spacecraft, Sensor, AoI, Coverage
from missiontools.plotting import plot_ground_track, plot_coverage_map

## 1. Build the spacecraft

In [ ]:
EPOCH = np.datetime64('2025-06-01T00:00:00', 'us')

sc = Spacecraft.sunsync(
    altitude_km=550.0,
    node_solar_time='10:30',
    epoch=EPOCH,
)

print(f"Semi-major axis : {sc.a/1e3:.1f} km")
print(f"Inclination     : {np.degrees(sc.i):.2f} deg")
print(f"RAAN            : {np.degrees(sc.raan):.2f} deg")
print(f"Propagator      : {sc.propagator_type}")

## 2. Attach a nadir-pointed sensor

In [ ]:
sensor = Sensor(half_angle_deg=20.0, body_vector=[0, 0, 1])
sc.add_sensor(sensor)

print(f"FOV half-angle  : {np.degrees(sensor.half_angle_rad):.1f}°")
print(f"Swath at nadir  : {2 * sc.a * np.tan(sensor.half_angle_rad) / 1e3:.0f} km (approx)")

## 3. Define the Area of Interest

In [ ]:
# 20 000 km² ≈ 2×10^10 m² per point
aoi = AoI.from_geography('Canada', point_density=20_000e6)
print(f"AoI sample points: {len(aoi)}")

## 4. Set up Coverage analysis

In [ ]:
cov = Coverage(
    aoi,
    [sensor],
    el_min_deg=20.0,
    sza_max_deg=70.0,
)

## 5. Run the 14-day simulation

In [ ]:
T_START = EPOCH
T_END   = EPOCH + np.timedelta64(14 * 24 * 3600, 's')

print("Computing coverage fraction ...")
frac = cov.coverage_fraction(T_START, T_END)
print(f"  Final cumulative coverage : {frac['final_cumulative']:.1%}")
print(f"  Mean instantaneous        : {frac['mean_fraction']:.1%}")

print("\nComputing revisit times ...")
rev = cov.revisit_time(T_START, T_END)

## 6. Ground track — first 24 hours

In [ ]:
fig = plt.figure(figsize=(14, 6))
ax  = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

plot_ground_track(
    sc,
    T_START,
    T_START + np.timedelta64(24 * 3600, 's'),
    ax=ax,
    color='tab:orange',
    label='550 km SSO 10:30',
    linewidth=0.8,
)
ax.set_title('Ground track — first 24 hours')
ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

## 7. Coverage maps over Canada

We use **EPSG:3347** (Statistics Canada Lambert), a conic projection optimised
for Canada.

In [ ]:
canada_proj = ccrs.epsg(3347)  # Statistics Canada Lambert

# --- Mean revisit time (hours) -------------------------------------------
mean_rev_hrs = rev['mean_revisit'] / 3600.0  # seconds → hours

fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': canada_proj},
)
plot_coverage_map(
    aoi,
    mean_rev_hrs,
    ax=ax,
    auto_window=True,
    cmap='plasma_r',
    colorbar_label='Mean revisit time (hours)',
    title='14-day mean revisit time — Canada\n550 km SSO 10:30 LTAN, 20° half-angle, el ≥ 20°, SZA < 70°',
)
plt.tight_layout()
plt.show()

In [ ]:
# --- Maximum revisit time (hours) -----------------------------------------
max_rev_hrs = rev['max_revisit'] / 3600.0  # seconds → hours

fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': canada_proj},
)
plot_coverage_map(
    aoi,
    max_rev_hrs,
    ax=ax,
    auto_window=True,
    cmap='plasma_r',
    colorbar_label='Max revisit time (hours)',
    title='14-day maximum revisit time — Canada\n550 km SSO 10:30 LTAN, 20° half-angle, el ≥ 20°, SZA < 70°',
)
plt.tight_layout()
plt.show()

## 8. Summary statistics

In [ ]:
print("=" * 50)
print("14-day coverage summary — Canada")
print("=" * 50)
print(f"Cumulative coverage fraction : {frac['final_cumulative']:.1%}")
print(f"Mean instantaneous coverage  : {frac['mean_fraction']:.1%}")
print()
global_mean_hrs = rev['global_mean'] / 3600.0
global_max_hrs  = rev['global_max']  / 3600.0
print(f"Global mean revisit time     : {global_mean_hrs:.1f} h")
print(f"Global max revisit time      : {global_max_hrs:.1f} h")